## # Data Inventory

This notebook performs an initial inventory of the myVal dataset.

The objective is to understand the workbook structure, available sheets, dataset dimensions, columns, missing values, duplicate records, and potential relationships before defining the modelling target.

In [1]:
from pathlib import Path

import pandas as pd


# -------------------------------------------------------------------
# Data Source
# -------------------------------------------------------------------

# Raw dataset provided for the project.
DATA_PATH = Path("../../data/raw/myVal_Synthetic_Datasets_release_v2.xlsx")

# Validate that the file exists before loading it.
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

print(f"Dataset found: {DATA_PATH}")

Dataset found: ..\data\raw\myVal_Synthetic_Datasets_release_v2.xlsx


## Workbook Structure

In [2]:
# -------------------------------------------------------------------
# Workbook Inspection
# -------------------------------------------------------------------

# Load the Excel workbook metadata without reading every sheet yet.
workbook = pd.ExcelFile(DATA_PATH)

# Retrieve all available sheet names.
sheet_names = workbook.sheet_names

print(f"Number of sheets: {len(sheet_names)}")
print("\nAvailable sheets:")

for i, sheet in enumerate(sheet_names, start=1):
    print(f"{i:02d}. {sheet}")

Number of sheets: 26

Available sheets:
01. Overview
02. Contents
03. Schema Verification
04. Data Dictionary
05. customers
06. customer_profiles
07. properties
08. base_categories
09. assets
10. asset_images
11. asset_attachments
12. ai_analysis
13. ai_raw_responses
14. valuations
15. property_videos
16. user_activity
17. policies
18. pds_documents
19. pds_sections
20. claims
21. claim_items
22. claim_events
23. notifications
24. user_feedback
25. ai_interactions
26. Sheet1


## Sheet Inventory

The following step summarises the size of each worksheet to identify the main data tables and supporting documentation contained in the workbook.

In [3]:
# -------------------------------------------------------------------
# Sheet Inventory
# -------------------------------------------------------------------

sheet_inventory = []

# Read each worksheet individually and record its dimensions.
for sheet in sheet_names:
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)

    sheet_inventory.append(
        {
            "sheet_name": sheet,
            "rows": df.shape[0],
            "columns": df.shape[1],
        }
    )

# Convert the inventory into a DataFrame for easier inspection.
inventory_df = pd.DataFrame(sheet_inventory)

inventory_df

,sheet_name,rows,columns
0,Overview,44,2
1,Contents,21,7
2,Schema Verification,15,4
3,Data Dictionary,332,9
4,customers,500,15
5,customer_profiles,500,15
6,properties,795,19
7,base_categories,7,9
8,assets,8309,26
9,asset_images,8309,13


## Selected Modelling Tables

The initial analysis focuses only on the datasets that can contribute directly to the next-category prediction task.

These tables describe customer and household context, the known inventory, outputs from existing AI processes, video-derived information, and the category structure used to define candidate predictions.

In [4]:
# -------------------------------------------------------------------
# Selected Modelling Tables
# -------------------------------------------------------------------

# Tables considered relevant to the next-category prediction problem.
MODELLING_SHEETS = [
    "customers",
    "customer_profiles",
    "properties",
    "base_categories",
    "assets",
    "asset_images",
    "ai_analysis",
    "property_videos",
]

# Load only the selected worksheets into memory.
data = {
    sheet: pd.read_excel(DATA_PATH, sheet_name=sheet)
    for sheet in MODELLING_SHEETS
}

# Confirm that every selected table was loaded successfully.
for table_name, df in data.items():
    print(
        f"{table_name:<20} "
        f"{df.shape[0]:>7,} rows × "
        f"{df.shape[1]:>3} columns"
    )

customers                500 rows ×  15 columns
customer_profiles        500 rows ×  15 columns
properties               795 rows ×  19 columns
base_categories            7 rows ×   9 columns
assets                 8,309 rows ×  26 columns
asset_images           8,309 rows ×  13 columns
ai_analysis           24,634 rows ×  25 columns
property_videos          219 rows ×  11 columns


## Table Structure

The structure of each selected dataset is inspected to understand available fields, candidate identifiers, and potential relationships between tables before any modelling dataset is constructed.

In [5]:
# -------------------------------------------------------------------
# Table Structure
# -------------------------------------------------------------------

# Display the column names for every selected dataset.
for table_name, df in data.items():

    print("=" * 80)
    print(f"{table_name.upper()}")
    print("=" * 80)

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

    for column in df.columns:
        print(f"- {column}")

    print()

CUSTOMERS
Shape: 500 rows × 15 columns

- Customer_ID
- Age_Band
- Gender
- City
- State
- Country
- Postcode
- Occupation
- Account_Created
- Membership_Level
- Role
- Auth_Provider
- Terms_Accepted
- Last_Active_At
- Account_Status

CUSTOMER_PROFILES
Shape: 500 rows × 15 columns

- Customer_ID
- Financial_Literacy_Band
- Insurance_Confidence_1_5
- Engagement_Tier
- Preferred_Channel
- Preferred_Explanation_Style
- Notification_Opt_In
- Accessibility_Need
- Language_At_Home
- Household_Type
- Income_Band
- Tenure_Months
- Prior_Claim_Experience
- Churn_Risk_Score
- Underinsurance_Awareness

PROPERTIES
Shape: 795 rows × 19 columns

- Property_ID
- Customer_ID
- Property_Type
- Occupancy_Type
- Number_Of_Occupants
- Number_Of_Bedrooms
- Address_Locality
- Address_State
- Address_Postcode
- Address_Country
- Timezone
- Latitude
- Longitude
- Property_Valuation_AUD
- Is_Home_Insured
- Insurance_Coverage_Type
- Insurance_Company
- Created_At
- Is_Primary

BASE_CATEGORIES
Shape: 7 rows × 9 

## Key And Relationship Validation

Before combining the selected datasets, the identifier structure and relationships between tables are validated.

This step verifies primary-key uniqueness, foreign-key coverage, and table cardinalities to ensure that future joins preserve the intended unit of observation and do not unintentionally duplicate records.

In [6]:
# -------------------------------------------------------------------
# Key And Relationship Validation
# -------------------------------------------------------------------

customers = data["customers"]
customer_profiles = data["customer_profiles"]
properties = data["properties"]
base_categories = data["base_categories"]
assets = data["assets"]
asset_images = data["asset_images"]
ai_analysis = data["ai_analysis"]
property_videos = data["property_videos"]


# -------------------------------------------------------------------
# 1. Primary Key Uniqueness
# -------------------------------------------------------------------

primary_keys = {
    "customers": ("Customer_ID", customers),
    "customer_profiles": ("Customer_ID", customer_profiles),
    "properties": ("Property_ID", properties),
    "base_categories": ("Category_ID", base_categories),
    "assets": ("Asset_ID", assets),
    "asset_images": ("Image_ID", asset_images),
    "ai_analysis": ("Analysis_ID", ai_analysis),
    "property_videos": ("Video_ID", property_videos),
}

print("PRIMARY KEY VALIDATION")
print("-" * 80)

for table_name, (key, df) in primary_keys.items():
    total_rows = len(df)
    unique_keys = df[key].nunique(dropna=False)
    duplicated_keys = df[key].duplicated().sum()
    missing_keys = df[key].isna().sum()

    print(
        f"{table_name:<20} "
        f"rows={total_rows:>6,} | "
        f"unique={unique_keys:>6,} | "
        f"duplicates={duplicated_keys:>4,} | "
        f"missing={missing_keys:>4,}"
    )

PRIMARY KEY VALIDATION
--------------------------------------------------------------------------------
customers            rows=   500 | unique=   500 | duplicates=   0 | missing=   0
customer_profiles    rows=   500 | unique=   500 | duplicates=   0 | missing=   0
properties           rows=   795 | unique=   795 | duplicates=   0 | missing=   0
base_categories      rows=     7 | unique=     7 | duplicates=   0 | missing=   0
assets               rows= 8,309 | unique= 8,309 | duplicates=   0 | missing=   0
asset_images         rows= 8,309 | unique= 8,309 | duplicates=   0 | missing=   0
ai_analysis          rows=24,634 | unique=24,634 | duplicates=   0 | missing=   0
property_videos      rows=   219 | unique=   219 | duplicates=   0 | missing=   0


In [7]:
# -------------------------------------------------------------------
# 2. Foreign Key Coverage
# -------------------------------------------------------------------

relationships = [
    # Child table, child key, parent table, parent key
    ("customer_profiles", customer_profiles, "Customer_ID",
     "customers", customers, "Customer_ID"),

    ("properties", properties, "Customer_ID",
     "customers", customers, "Customer_ID"),

    ("assets", assets, "Customer_ID",
     "customers", customers, "Customer_ID"),

    ("assets", assets, "Property_ID",
     "properties", properties, "Property_ID"),

    ("assets", assets, "Category_ID",
     "base_categories", base_categories, "Category_ID"),

    ("asset_images", asset_images, "Asset_ID",
     "assets", assets, "Asset_ID"),

    ("ai_analysis", ai_analysis, "Asset_ID",
     "assets", assets, "Asset_ID"),

    ("ai_analysis", ai_analysis, "Image_ID",
     "asset_images", asset_images, "Image_ID"),

    ("property_videos", property_videos, "Property_ID",
     "properties", properties, "Property_ID"),
]

print("FOREIGN KEY COVERAGE")
print("-" * 100)

for (
    child_name,
    child_df,
    child_key,
    parent_name,
    parent_df,
    parent_key,
) in relationships:

    # Null foreign keys are reported separately because some relationships
    # may legitimately be optional.
    non_null_values = child_df[child_key].dropna()

    unmatched = ~non_null_values.isin(parent_df[parent_key])
    unmatched_count = unmatched.sum()

    print(
        f"{child_name}.{child_key:<18} -> "
        f"{parent_name}.{parent_key:<15} | "
        f"non-null={len(non_null_values):>6,} | "
        f"unmatched={unmatched_count:>5,} | "
        f"null={child_df[child_key].isna().sum():>5,}"
    )

FOREIGN KEY COVERAGE
----------------------------------------------------------------------------------------------------
customer_profiles.Customer_ID        -> customers.Customer_ID     | non-null=   500 | unmatched=    0 | null=    0
properties.Customer_ID        -> customers.Customer_ID     | non-null=   795 | unmatched=    0 | null=    0
assets.Customer_ID        -> customers.Customer_ID     | non-null= 8,309 | unmatched=    0 | null=    0
assets.Property_ID        -> properties.Property_ID     | non-null= 8,309 | unmatched=    0 | null=    0
assets.Category_ID        -> base_categories.Category_ID     | non-null= 8,309 | unmatched=    0 | null=    0
asset_images.Asset_ID           -> assets.Asset_ID        | non-null= 8,309 | unmatched=    0 | null=    0
ai_analysis.Asset_ID           -> assets.Asset_ID        | non-null=24,634 | unmatched=    0 | null=    0
ai_analysis.Image_ID           -> asset_images.Image_ID        | non-null=24,634 | unmatched=    0 | null=    0
property_vi

In [8]:
# -------------------------------------------------------------------
# 3. Relationship Cardinalities
# -------------------------------------------------------------------

print("RELATIONSHIP CARDINALITIES")
print("-" * 80)

# Number of properties associated with each customer.
properties_per_customer = properties.groupby("Customer_ID")["Property_ID"].nunique()

# Number of assets associated with each property.
assets_per_property = assets.groupby("Property_ID")["Asset_ID"].nunique()

# Number of images associated with each asset.
images_per_asset = asset_images.groupby("Asset_ID")["Image_ID"].nunique()

# Number of AI analysis rows associated with each asset.
ai_rows_per_asset = ai_analysis.groupby("Asset_ID")["Analysis_ID"].nunique()

# Number of videos associated with each property.
videos_per_property = property_videos.groupby("Property_ID")["Video_ID"].nunique()


def describe_relationship(name, series):
    """Print a compact summary of relationship multiplicity."""
    print(
        f"{name:<30} "
        f"min={series.min():>5.0f} | "
        f"median={series.median():>5.1f} | "
        f"mean={series.mean():>6.2f} | "
        f"max={series.max():>5.0f}"
    )


describe_relationship("Properties per customer", properties_per_customer)
describe_relationship("Assets per property", assets_per_property)
describe_relationship("Images per asset", images_per_asset)
describe_relationship("AI rows per asset", ai_rows_per_asset)
describe_relationship("Videos per property", videos_per_property)

RELATIONSHIP CARDINALITIES
--------------------------------------------------------------------------------
Properties per customer        min=    1 | median=  1.0 | mean=  1.59 | max=    6
Assets per property            min=    1 | median=  2.0 | mean= 10.45 | max=  470
Images per asset               min=    1 | median=  1.0 | mean=  1.00 | max=    1
AI rows per asset              min=    1 | median=  3.0 | mean=  3.00 | max=    3
Videos per property            min=    1 | median=  1.0 | mean=  1.30 | max=    3


In [9]:
# -------------------------------------------------------------------
# 4. AI Analysis Candidate Structure
# -------------------------------------------------------------------

# Inspect how many candidate predictions are generated for each analysis source.
candidate_rank_counts = (
    ai_analysis["Candidate_Rank"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Candidate Rank Distribution")
print("-" * 40)
print(candidate_rank_counts)

print("\nUnique AI analyses:", f"{ai_analysis['Analysis_ID'].nunique():,}")
print("Unique assets analysed:", f"{ai_analysis['Asset_ID'].nunique():,}")
print("Unique images analysed:", f"{ai_analysis['Image_ID'].nunique():,}")

Candidate Rank Distribution
----------------------------------------
Candidate_Rank
1    8224
2    8216
3    8194
Name: count, dtype: int64

Unique AI analyses: 24,634
Unique assets analysed: 8,224
Unique images analysed: 8,224


## Data Source Coverage

The availability of upstream evidence is evaluated across properties and assets.

This step determines how frequently AI analysis and property video information are available, since these signals may become important inputs to the next-category prediction model but are not necessarily present for every household.

In [10]:
# -------------------------------------------------------------------
# Data Source Coverage
# -------------------------------------------------------------------

# Asset-level AI coverage.
assets_with_ai = ai_analysis["Asset_ID"].nunique()
total_assets = assets["Asset_ID"].nunique()

ai_coverage_pct = assets_with_ai / total_assets * 100


# Property-level video coverage.
properties_with_video = property_videos["Property_ID"].nunique()
total_properties = properties["Property_ID"].nunique()

video_coverage_pct = properties_with_video / total_properties * 100


print("DATA SOURCE COVERAGE")
print("-" * 60)

print(
    f"Assets with AI analysis: "
    f"{assets_with_ai:,} / {total_assets:,} "
    f"({ai_coverage_pct:.2f}%)"
)

print(
    f"Properties with video:    "
    f"{properties_with_video:,} / {total_properties:,} "
    f"({video_coverage_pct:.2f}%)"
)

DATA SOURCE COVERAGE
------------------------------------------------------------
Assets with AI analysis: 8,224 / 8,309 (98.98%)
Properties with video:    169 / 795 (21.26%)


In [11]:
# -------------------------------------------------------------------
# AI Candidate Completeness
# -------------------------------------------------------------------

# Count the number of AI candidates available for each analysed asset.
ai_candidates_per_asset = (
    ai_analysis
    .groupby("Asset_ID")["Candidate_Rank"]
    .nunique()
)

candidate_completeness = (
    ai_candidates_per_asset
    .value_counts()
    .sort_index()
)

print("AI CANDIDATES PER ASSET")
print("-" * 40)

for number_candidates, number_assets in candidate_completeness.items():
    print(
        f"{number_candidates} candidate(s): "
        f"{number_assets:,} assets"
    )

AI CANDIDATES PER ASSET
----------------------------------------
1 candidate(s): 8 assets
2 candidate(s): 22 assets
3 candidate(s): 8,194 assets


## Household Category Structure

The category structure of the documented household inventory is examined to understand the prediction space and how categories are distributed across properties.

This provides the foundation for determining whether the available inventory can support a meaningful next-category prediction task.

In [12]:
# -------------------------------------------------------------------
# Category Universe
# -------------------------------------------------------------------

# Display the complete category set defined by myVal.
category_universe = (
    base_categories[
        [
            "Category_ID",
            "Name",
            "Display_Order",
            "Typical_Depreciation_Rate_PA",
            "Item_Limit_AUD",
            "Category_Limit_AUD",
        ]
    ]
    .sort_values("Display_Order")
    .reset_index(drop=True)
)

category_universe

,Category_ID,Name,Display_Order,Typical_Depreciation_Rate_PA,Item_Limit_AUD,Category_Limit_AUD
0,CAT-001,Electronics,1,0.20,1000.0,5000.0
1,CAT-002,Jewellery and Personal Items,2,0.02,1000.0,5000.0
2,CAT-003,Furniture,3,0.08,NaN,NaN
3,CAT-004,Clothings,4,0.25,1000.0,NaN
4,CAT-005,Appliances,5,0.12,1000.0,NaN
5,CAT-006,Other Items,6,0.12,1000.0,NaN
6,CAT-007,Vehicles,7,0.11,NaN,NaN


### Category Representation In The Asset Inventory

In [13]:
# -------------------------------------------------------------------
# Asset Distribution By Category
# -------------------------------------------------------------------

# Count documented assets in each category.
category_asset_counts = (
    assets
    .groupby(["Category_ID", "Category"])
    .agg(
        asset_count=("Asset_ID", "nunique"),
        property_count=("Property_ID", "nunique"),
    )
    .reset_index()
)

# Calculate how much of the total asset inventory each category represents.
category_asset_counts["asset_percentage"] = (
    category_asset_counts["asset_count"]
    / assets["Asset_ID"].nunique()
    * 100
)

category_asset_counts = category_asset_counts.sort_values(
    "asset_count",
    ascending=False,
)

category_asset_counts

,Category_ID,Category,asset_count,property_count,asset_percentage
0,CAT-001,Electronics,2637,621,31.736671
2,CAT-003,Furniture,2056,246,24.744253
4,CAT-005,Appliances,1163,234,13.996871
5,CAT-006,Other Items,1120,152,13.479360
1,CAT-002,Jewellery and Personal Items,946,202,11.385245
3,CAT-004,Clothings,365,59,4.392827
6,CAT-007,Vehicles,22,16,0.264773


### Categories Per Property

In [14]:
# -------------------------------------------------------------------
# Category Coverage Per Property
# -------------------------------------------------------------------

# Count how many distinct categories are documented for each property.
categories_per_property = (
    assets
    .groupby("Property_ID")["Category_ID"]
    .nunique()
)

print("CATEGORIES PER PROPERTY")
print("-" * 50)

print(f"Properties with assets: {categories_per_property.shape[0]:,}")
print(f"Minimum categories:     {categories_per_property.min():.0f}")
print(f"Median categories:      {categories_per_property.median():.1f}")
print(f"Mean categories:        {categories_per_property.mean():.2f}")
print(f"Maximum categories:     {categories_per_property.max():.0f}")

CATEGORIES PER PROPERTY
--------------------------------------------------
Properties with assets: 795
Minimum categories:     1
Median categories:      1.0
Mean categories:        1.92
Maximum categories:     7


In [15]:
# -------------------------------------------------------------------
# Distribution Of Category Coverage
# -------------------------------------------------------------------

category_coverage_distribution = (
    categories_per_property
    .value_counts()
    .sort_index()
    .rename_axis("number_of_categories")
    .reset_index(name="number_of_properties")
)

category_coverage_distribution["percentage_of_properties"] = (
    category_coverage_distribution["number_of_properties"]
    / categories_per_property.shape[0]
    * 100
)

category_coverage_distribution

,number_of_categories,number_of_properties,percentage_of_properties
0,1,440,55.345912
1,2,174,21.886792
2,3,74,9.308176
3,4,42,5.283019
4,5,40,5.031447
5,6,23,2.893082
6,7,2,0.251572


## Dataset Semantics Verification

Before defining the prediction target, the dataset documentation is reviewed to determine whether the asset inventory represents the complete synthetic household or only the portion of the household already documented by the customer.

This distinction is essential because the absence of a category can only be treated as a missing target if the underlying complete household inventory is known.

In [16]:
# -------------------------------------------------------------------
# Dataset Documentation
# -------------------------------------------------------------------

# Load documentation sheets that describe the meaning and generation
# of the underlying datasets.
overview = pd.read_excel(DATA_PATH, sheet_name="Overview")
data_dictionary = pd.read_excel(DATA_PATH, sheet_name="Data Dictionary")
schema_verification = pd.read_excel(DATA_PATH, sheet_name="Schema Verification")

print("OVERVIEW")
print("-" * 80)
display(overview)

print("\nDATA DICTIONARY")
print("-" * 80)
display(data_dictionary.head(20))

print("\nSCHEMA VERIFICATION")
print("-" * 80)
display(schema_verification.head(20))

OVERVIEW
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1
0,NaN,myVal — Synthetic Datasets
1,NaN,iLab Challenge — Reimagining Insurance Through AI
2,NaN,NaN
3,NaN,Purpose
4,NaN,Fully synthetic datasets that mirror the struc...
5,NaN,Verified against the live production schema in...
6,NaN,Frontend coverage was checked against the Flut...
7,NaN,NaN
8,NaN,Which copy is this
9,NaN,RELEASE COPY — for the university. The six fie...



DATA DICTIONARY
--------------------------------------------------------------------------------


,Dataset,Column Name,Type,Allowed Values / Format,Description,Schema Status,Frontend Availability,Decision,Decision Rationale
0,customers,Customer_ID,Text,CUS-00001,Primary key. Join key for every other dataset.,Derived,Mobile (record ID; synthetic export format dif...,Include,Derived or synthetic join field needed to anal...
1,customers,First_Name,Text,NaN,"Synthetic given name, randomly paired with a f...",Production,Mobile,Exclude,Not needed for challenge analysis; remove dire...
2,customers,Last_Name,Text,NaN,Synthetic family name.,Production,Mobile,Exclude,Not needed for challenge analysis; remove dire...
3,customers,Date_of_Birth,Date,YYYY-MM-DD,"Synthetic DOB, age distribution centred on 43.",Production,Mobile,Exclude,Use Age_Band instead to minimise unnecessary p...
4,customers,Age_Band,Text,18-24 / 25-34 / 35-44 / 45-54 / 55-64 / 65+,Broad band derived from DOB. Prefer this over ...,Derived,Not shown — analytical or join field,Include,Derived or synthetic join field needed to anal...
5,customers,Gender,Text,Female / Male / Prefer not to say,Exact values sent by the active mobile GenderT...,Production,Mobile,Include,Current production business field represented ...
6,customers,Email,Text,name.surname{n}@example.com,RFC 2606 reserved domain — cannot route to a r...,Production,Mobile,Exclude,Not needed for modelling; exclude contact deta...
7,customers,Phone,Text,+61 2 5550 0000,Drawn from the geographic blocks ACMA permanen...,Production,Mobile,Exclude,Not needed for modelling; exclude contact deta...
8,customers,City,Text,50 AU suburbs,Suburb of residence. Public geographic data.,Proposed,Not in current frontend — proposed field,Optional,Proposed schema addition; include only when th...
9,customers,State,Text,NSW / VIC / QLD / WA / SA / TAS / ACT / NT,Australian state or territory.,Proposed,Not in current frontend — proposed field,Optional,Proposed schema addition; include only when th...



SCHEMA VERIFICATION
--------------------------------------------------------------------------------


,#,Workbook says,Production reality,Resolution in this dataset
0,1.0,Asset joins straight to Customer_ID,User → Property → PropertyItem. An item cannot...,assets.csv carries BOTH Customer_ID and Proper...
1,2.0,No Property column list at all,Property is the richest table and holds every ...,properties.csv adds the production business fi...
2,3.0,Attachment keyed to Customer_ID,PropertyItemAttachment.propertyItemId — attach...,"asset_attachments.csv keys on Asset_ID, with C..."
3,4.0,Valuation = per-asset history,The production Valuation entity is a CONTENTS-...,Both shapes produced: valuations.csv (per-asse...
4,5.0,Customer has City / State / Postcode / Occupat...,The User entity has none of these. Location li...,Kept on customers.csv because the challenge ne...
5,6.0,Asset has Brand / Model / Year / Ownership_Sta...,"PropertyItem has none of them. It has name, va...",Generated anyway (the brief calls for asset cl...
6,7.0,"No Insurance, AI_Analysis or video entity colu...",Insurance is denormalised onto Property; AI an...,"policies.csv, ai_analysis.csv and property_vid..."
7,8.0,ER diagram draws Insurance → Asset,No direct production relationship exists. An a...,assets.csv carries a denormalised Policy_ID fo...
8,9.0,ER diagram feeds AI_Analysis from both Image a...,Production document extraction lives on the at...,ai_analysis.csv represents both sources and di...
9,10.0,Added production datasets are complete product...,"They are selected analytical views, not databa...",These omissions are intentional: infrastructur...


### Target-Relevant Documentation

In [17]:
# -------------------------------------------------------------------
# Search Dataset Documentation For Target-Relevant Concepts
# -------------------------------------------------------------------

# Terms related to inventory completeness, captured evidence,
# categories and the generation of household assets.
SEARCH_TERMS = [
    "asset",
    "inventory",
    "category",
    "complete",
    "missing",
    "documented",
    "synthetic",
    "property",
]


def search_documentation(df, terms):
    """
    Return rows containing any target-relevant term across
    any textual column in the documentation table.
    """

    # Convert every value to string so all columns can be searched.
    searchable = df.astype(str)

    row_mask = searchable.apply(
        lambda row: row.str.contains(
            "|".join(terms),
            case=False,
            na=False,
            regex=True,
        ).any(),
        axis=1,
    )

    return df.loc[row_mask]


overview_matches = search_documentation(overview, SEARCH_TERMS)
dictionary_matches = search_documentation(data_dictionary, SEARCH_TERMS)

print("OVERVIEW MATCHES")
display(overview_matches)

print("\nDATA DICTIONARY MATCHES")
display(dictionary_matches)

OVERVIEW MATCHES


,Unnamed: 0,Unnamed: 1
0,NaN,myVal — Synthetic Datasets
4,NaN,Fully synthetic datasets that mirror the struc...
17,NaN,5. Keep all joins based on the synthetic IDs (...
24,NaN,"ITEM DATA IS REAL AI OUTPUT, NOT GENERATED. As..."
25,NaN,NO IMAGES OR DOCUMENTS SHIP. Image_URL and Doc...
29,NaN,├── Property (Property_ID) ── Policy (Polic...
30,NaN,│ ├── Asset (Asset_ID) ── Image (Ima...
33,NaN,│ └── Property_Video (Video_ID)
38,NaN,NOTE: the original workbook joined Asset direc...
43,NaN,"500 customers · 795 properties · 8,309 assets ..."



DATA DICTIONARY MATCHES


,Dataset,Column Name,Type,Allowed Values / Format,Description,Schema Status,Frontend Availability,Decision,Decision Rationale
0,customers,Customer_ID,Text,CUS-00001,Primary key. Join key for every other dataset.,Derived,Mobile (record ID; synthetic export format dif...,Include,Derived or synthetic join field needed to anal...
1,customers,First_Name,Text,NaN,"Synthetic given name, randomly paired with a f...",Production,Mobile,Exclude,Not needed for challenge analysis; remove dire...
2,customers,Last_Name,Text,NaN,Synthetic family name.,Production,Mobile,Exclude,Not needed for challenge analysis; remove dire...
3,customers,Date_of_Birth,Date,YYYY-MM-DD,"Synthetic DOB, age distribution centred on 43.",Production,Mobile,Exclude,Use Age_Band instead to minimise unnecessary p...
4,customers,Age_Band,Text,18-24 / 25-34 / 35-44 / 45-54 / 55-64 / 65+,Broad band derived from DOB. Prefer this over ...,Derived,Not shown — analytical or join field,Include,Derived or synthetic join field needed to anal...
...,...,...,...,...,...,...,...,...,...
305,user_feedback,Confusion_Signal,Boolean,TRUE / FALSE,The customer has said outright that something ...,Derived,Not shown — analytical or join field,Include,Derived or synthetic join field needed to anal...
317,ai_interactions,Assistant_Response,Text,NaN,Answers THIS question — the answer text is wri...,Extension,Not in current frontend — challenge-only,Include,Challenge extension required for the insurance...
318,ai_interactions,Grounded_On_Type,Text,pds_section / policy / asset / none,"What the answer was retrieved against. ""none"" ...",Extension,Not in current frontend — challenge-only,Include,Challenge extension required for the insurance...
321,ai_interactions,Literacy_Band_Target,Text,low / medium / high,The customer's band.,Derived,Not shown — analytical or join field,Include,Derived or synthetic join field needed to anal...


### Asset Inventory Semantics

In [18]:
# -------------------------------------------------------------------
# Search Documentation For Asset Inventory Meaning
# -------------------------------------------------------------------

TARGET_TERMS = [
    "complete",
    "documented",
    "captured",
    "inventory",
    "assets",
    "household",
]

for table_name, df in {
    "Overview": overview,
    "Data Dictionary": data_dictionary,
}.items():

    print(f"\n{table_name}")
    print("-" * 80)

    searchable = df.astype(str)

    mask = searchable.apply(
        lambda row: row.str.contains(
            "|".join(TARGET_TERMS),
            case=False,
            na=False,
            regex=True,
        ).any(),
        axis=1,
    )

    display(df.loc[mask].head(30))


Overview
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1
24,NaN,"ITEM DATA IS REAL AI OUTPUT, NOT GENERATED. As..."
43,NaN,"500 customers · 795 properties · 8,309 assets ..."



Data Dictionary
--------------------------------------------------------------------------------


,Dataset,Column Name,Type,Allowed Values / Format,Description,Schema Status,Frontend Availability,Decision,Decision Rationale
23,customer_profiles,Engagement_Tier,Text,dormant / casual / engaged / power,"Drives inventory depth, session count and atta...",Extension,Not in current frontend — challenge-only,Include,Challenge extension required for the insurance...
29,customer_profiles,Household_Type,Text,5 values,Household composition.,Extension,Not in current frontend — challenge-only,Include,Challenge extension required for the insurance...
49,properties,Property_Valuation_AUD,Currency,AUD,Production Property.valuation: estimated house...,Production,Mobile,Include,Current production business field represented ...
62,assets,Asset_ID,Text,AST-0000001,Primary key.,Derived,Mobile,Include,Derived or synthetic join field needed to anal...
63,assets,Customer_ID,Text,NaN,Denormalised for convenience.,Derived,Not shown — analytical or join field,Include,Derived or synthetic join field needed to anal...
64,assets,Property_ID,Text,NaN,Production PropertyItem.propertyId. REQUIRED —...,Production,Mobile,Include,Current production business field represented ...
65,assets,Policy_ID,Text,NaN,Implements the Insurance → Asset edge on the w...,Extension,Not in current frontend — challenge-only,Include,Challenge extension required for the insurance...
66,assets,Category_ID,Text,NaN,Production PropertyItem.categoryId.,Production,Mobile,Include,Current production business field represented ...
67,assets,Category,Text,26 of the 29 app categories,Category name. Matches the workbook Asset.Cate...,Derived,Not shown — analytical or join field,Include,Derived or synthetic join field needed to anal...
68,assets,Asset_Name,Text,NaN,Production PropertyItem.name. Reformatted from...,Production,Mobile,Include,Current production business field represented ...


## Asset Documentation Sequence

The asset creation timestamps are inspected to determine whether the dataset contains a usable sequence of household documentation events.

If the timestamps preserve the order in which categories were added, they can provide an observed target for modelling the next category a customer is likely to document.

In [19]:
# -------------------------------------------------------------------
# Asset Documentation Sequence
# -------------------------------------------------------------------

# Convert creation timestamps to datetime to ensure correct ordering.
assets["Created_At"] = pd.to_datetime(
    assets["Created_At"],
    errors="coerce",
)

print("ASSET CREATION TIMESTAMPS")
print("-" * 60)

print(f"Total assets:            {len(assets):,}")
print(f"Missing timestamps:      {assets['Created_At'].isna().sum():,}")
print(f"Unique timestamps:       {assets['Created_At'].nunique():,}")
print(f"Earliest timestamp:      {assets['Created_At'].min()}")
print(f"Latest timestamp:        {assets['Created_At'].max()}")

ASSET CREATION TIMESTAMPS
------------------------------------------------------------
Total assets:            8,309
Missing timestamps:      0
Unique timestamps:       7,407
Earliest timestamp:      2024-09-19 06:19:28
Latest timestamp:        2026-09-13 17:45:06


### Property-Level Documentation History

The asset creation history is examined at property level to determine how many households contain multiple documentation events over time.

A usable next-category target requires properties with at least two distinct documentation moments so that a previous household state and a subsequent category can be observed.


In [20]:
# -------------------------------------------------------------------
# Property-Level Documentation History
# -------------------------------------------------------------------

# Summarise the documentation history available for each property.
property_history = (
    assets
    .groupby("Property_ID")
    .agg(
        total_assets=("Asset_ID", "nunique"),
        unique_categories=("Category_ID", "nunique"),
        documentation_events=("Created_At", "nunique"),
    )
)

# Identify properties that contain more than one documentation event.
properties_with_sequence = (
    property_history["documentation_events"] >= 2
).sum()

total_properties = property_history.shape[0]

sequence_coverage_pct = (
    properties_with_sequence / total_properties * 100
)

print("PROPERTY-LEVEL DOCUMENTATION HISTORY")
print("-" * 60)

print(f"Total properties:                 {total_properties:,}")
print(f"Properties with 2+ events:        {properties_with_sequence:,}")
print(f"Sequence coverage:                {sequence_coverage_pct:.2f}%")

print()
print("Documentation events per property")
print("-" * 60)

print(f"Minimum:                          {property_history['documentation_events'].min():.0f}")
print(f"Median:                           {property_history['documentation_events'].median():.1f}")
print(f"Mean:                             {property_history['documentation_events'].mean():.2f}")
print(f"Maximum:                          {property_history['documentation_events'].max():.0f}")

PROPERTY-LEVEL DOCUMENTATION HISTORY
------------------------------------------------------------
Total properties:                 795
Properties with 2+ events:        410
Sequence coverage:                51.57%

Documentation events per property
------------------------------------------------------------
Minimum:                          1
Median:                           2.0
Mean:                             9.35
Maximum:                          470


### Cross-Category Documentation Sequences

The documentation history is evaluated to determine how many properties contain multiple categories over time.

These properties are particularly relevant for the next-category prediction task because they provide observed transitions from one documented category to another.

In [21]:
# -------------------------------------------------------------------
# Cross-Category Documentation Sequences
# -------------------------------------------------------------------

# Count distinct documented categories for each property.
categories_per_property = (
    assets
    .groupby("Property_ID")["Category_ID"]
    .nunique()
)

# Properties with at least two different documented categories can
# provide observed cross-category transitions.
properties_with_category_sequence = (
    categories_per_property >= 2
).sum()

total_properties = categories_per_property.shape[0]

category_sequence_pct = (
    properties_with_category_sequence / total_properties * 100
)

print("CROSS-CATEGORY DOCUMENTATION COVERAGE")
print("-" * 60)

print(f"Total properties:                  {total_properties:,}")
print(f"Properties with 2+ categories:     {properties_with_category_sequence:,}")
print(f"Cross-category coverage:           {category_sequence_pct:.2f}%")

CROSS-CATEGORY DOCUMENTATION COVERAGE
------------------------------------------------------------
Total properties:                  795
Properties with 2+ categories:     355
Cross-category coverage:           44.65%


### Simultaneous Category Events

Asset timestamps are checked for simultaneous additions of multiple categories within the same property.

This validation is required before reconstructing category sequences, since multiple categories recorded at exactly the same time would make their internal ordering ambiguous.

In [22]:
# -------------------------------------------------------------------
# Simultaneous Category Events
# -------------------------------------------------------------------

# Count how many distinct categories were recorded at each timestamp
# within each property.
timestamp_category_counts = (
    assets
    .groupby(["Property_ID", "Created_At"])["Category_ID"]
    .nunique()
)

# Events containing more than one category have no reliable
# internal ordering based on Created_At alone.
ambiguous_events = timestamp_category_counts[
    timestamp_category_counts > 1
]

total_events = len(timestamp_category_counts)

print("SIMULTANEOUS CATEGORY EVENTS")
print("-" * 60)

print(f"Total property-time events:        {total_events:,}")
print(f"Multi-category events:             {len(ambiguous_events):,}")
print(
    f"Ambiguous event percentage:        "
    f"{len(ambiguous_events) / total_events * 100:.2f}%"
)

SIMULTANEOUS CATEGORY EVENTS
------------------------------------------------------------
Total property-time events:        7,437
Multi-category events:             184
Ambiguous event percentage:        2.47%


### Category Transition Patterns

Observed category transitions are extracted from the asset creation sequence to identify which documented categories are commonly followed by others.

This provides an initial behavioural view of how customers progress through household documentation and helps assess whether a next-category recommendation is learnable from the available data.

In [23]:
# -------------------------------------------------------------------
# Category Transition Patterns
# -------------------------------------------------------------------

# Keep only the fields required to reconstruct the documentation order.
sequence_df = (
    assets[
        [
            "Property_ID",
            "Created_At",
            "Category_ID",
            "Category",
        ]
    ]
    .sort_values(["Property_ID", "Created_At"])
    .copy()
)

# Remove duplicate category entries occurring at the same timestamp
# within the same property.
sequence_df = sequence_df.drop_duplicates(
    subset=["Property_ID", "Created_At", "Category_ID"]
)

# Exclude timestamps where more than one category was added at once,
# since the internal order between those categories is ambiguous.
event_category_counts = (
    sequence_df
    .groupby(["Property_ID", "Created_At"])["Category_ID"]
    .transform("nunique")
)

sequence_df = sequence_df[event_category_counts == 1].copy()

# Create the next documented category within each property.
sequence_df["Next_Category"] = (
    sequence_df
    .groupby("Property_ID")["Category"]
    .shift(-1)
)

# Keep only observed transitions.
transitions = sequence_df.dropna(subset=["Next_Category"])

# Count category-to-category transitions.
transition_counts = (
    transitions
    .groupby(["Category", "Next_Category"])
    .size()
    .reset_index(name="transition_count")
    .sort_values("transition_count", ascending=False)
)

transition_counts.head(20)

,Category,Next_Category,transition_count
23,Furniture,Furniture,1103
15,Electronics,Electronics,1096
39,Other Items,Other Items,589
31,Jewellery and Personal Items,Jewellery and Personal Items,519
0,Appliances,Appliances,409
16,Electronics,Furniture,282
22,Furniture,Electronics,255
8,Clothings,Clothings,224
25,Furniture,Other Items,207
37,Other Items,Furniture,193


### Cross-Category Transition Patterns

Same-category transitions are excluded to focus specifically on observed movements between different household categories.

These transitions are more relevant to the recommendation task because they represent cases where the customer moved from documenting one category to documenting a different category next.

In [24]:
# -------------------------------------------------------------------
# Cross-Category Transition Patterns
# -------------------------------------------------------------------

# Keep only transitions where the next documented category
# differs from the current category.
cross_category_transitions = transition_counts[
    transition_counts["Category"] != transition_counts["Next_Category"]
].copy()

# Display the most frequent observed category changes.
print(
    cross_category_transitions
    .head(20)
    .to_string(index=False)
)

                    Category                Next_Category  transition_count
                 Electronics                    Furniture               282
                   Furniture                  Electronics               255
                   Furniture                  Other Items               207
                 Other Items                    Furniture               193
                 Electronics                   Appliances               191
                  Appliances                  Electronics               185
                 Other Items                  Electronics               130
                 Electronics                  Other Items               127
                   Furniture                   Appliances               124
                  Appliances                    Furniture               123
                 Electronics Jewellery and Personal Items               111
Jewellery and Personal Items                  Electronics                98
Jewellery an

### Initial Finding

The observed documentation history contains meaningful cross-category transition patterns. Electronics, Furniture, Appliances and Other Items frequently appear in sequential transitions, indicating that the next documented category is not completely random.

However, the transitions are not strictly directional, which suggests that category history alone is unlikely to be sufficient for prediction. Household, property, customer, inventory and AI-derived context should therefore be incorporated into the modelling dataset.

## Data Inventory Summary

The initial data inventory confirms that the myVal dataset contains sufficient relational, temporal and AI-derived information to support the next-category recommendation problem.

The modelling context can be constructed using customer and household information, documented assets, existing AI analysis outputs and optional property-video signals.

Importantly, the `assets` table represents items already documented by the customer rather than a complete ground-truth household inventory. Therefore, an absent category cannot be interpreted as a category that is known to be missing.

Instead, the prediction problem is framed as recommending the category most likely to be relevant for the customer to document next.

Asset creation timestamps provide a usable documentation history. Cross-category transitions are observed across the dataset, with recurring movements between categories such as Electronics, Furniture, Appliances and Other Items. These patterns indicate that the next documented category is not completely random and may be predictable when combined with broader household and AI-derived context.

The next stage will formally define how historical household states and subsequent documented categories are transformed into supervised learning examples while preventing information from future events from leaking into the predictors.